In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timedelta

In [0]:
dbutils.widgets.text("start_date", "", "Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "", "End Date (YYYY-MM-DD)")
dbutils.widgets.text("tipo_carga", "incremental", "Tipo de carga")

# Obtener parámetros de los widgets (en UTC)
start_date_str = dbutils.widgets.get("start_date")  # ej. '2026-04-06T00:00:00'
end_date_str   = dbutils.widgets.get("end_date")    # ej. '2026-04-06T23:59:59'

# Convertir a objetos datetime
start_date_utc = datetime.fromisoformat(start_date_str)
end_date_utc   = datetime.fromisoformat(end_date_str)

# Ajustar a Lima (UTC-5)
start_date = start_date_utc - timedelta(hours=5)
end_date   = end_date_utc - timedelta(hours=5)

# Parámetros de fechas (strings)
start_date_str = start_date.strftime("%Y-%m-%d")
end_date_str   = end_date.strftime("%Y-%m-%d")

print("Start date Lima:", start_date_str)
print("End date Lima:", end_date_str)


# parametro de tipo de carga
tipo_carga = dbutils.widgets.get("tipo_carga")


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.gold.fact_contaminacion (
    fecha DATE COMMENT 'Fecha en formato yyyy-MM-dd',
    id_ubigeo STRING COMMENT 'Código único de distrito según INEI',
    cant_prom_pm2_5 DOUBLE COMMENT 'Promedio diario de concentración de PM2.5 (μg/m3)',
    cant_prom_pm10 DOUBLE COMMENT 'Promedio diario de concentración de PM10 (μg/m3)',
    cant_prom_co DOUBLE COMMENT 'Promedio diario de concentración de CO (Carbon monoxide), μg/m3',
    cant_prom_no2 DOUBLE COMMENT 'Promedio diario de concentración de NO2 (Nitrogen dioxide), μg/m3',
    cant_prom_so2 DOUBLE COMMENT 'Promedio diario de concentración de SO2 (Sulphur dioxide), μg/m3',
    cant_prom_o3 DOUBLE COMMENT 'Promedio diario de concentración de O3 (Ozone), μg/m3',
    moda_index_air_quality STRING COMMENT 'Moda diaria del índice de calidad del aire (Bueno, Aceptable, Moderado, Malo, Muy Malo)',
    fecha_actualizacion DATE COMMENT 'Fecha de actualización de la tabla Gold'
) USING DELTA
PARTITIONED BY (fecha);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.gold.fact_contaminacion_enriquecido (
    fecha DATE COMMENT 'Fecha en formato yyyy-MM-dd',
    id_ubigeo STRING COMMENT 'Código único de distrito según INEI',
    cant_prom_pm2_5 DOUBLE COMMENT 'Promedio diario de concentración de PM2.5 (μg/m3)',
    cant_prom_pm10 DOUBLE COMMENT 'Promedio diario de concentración de PM10 (μg/m3)',
    cant_prom_co DOUBLE COMMENT 'Promedio diario de concentración de CO (Carbon monoxide), μg/m3',
    cant_prom_no2 DOUBLE COMMENT 'Promedio diario de concentración de NO2 (Nitrogen dioxide), μg/m3',
    cant_prom_so2 DOUBLE COMMENT 'Promedio diario de concentración de SO2 (Sulphur dioxide), μg/m3',
    cant_prom_o3 DOUBLE COMMENT 'Promedio diario de concentración de O3 (Ozone), μg/m3',
    moda_index_air_quality STRING COMMENT 'Moda diaria del índice de calidad del aire (Bueno, Aceptable, Moderado, Malo, Muy Malo)',
    poblacion INT COMMENT 'Población total del distrito',
    superficie DOUBLE COMMENT 'Superficie del distrito en km²',
    densidad_poblacional DOUBLE COMMENT 'Número de habitantes por km² (población/superficie)',
    contaminacion_per_capita_1000 DOUBLE COMMENT 'Exposición promedio de PM2.5 por cada 1,000 habitantes',
    contaminacion_por_km2 DOUBLE COMMENT 'Concentración de PM2.5 por km² (PM2.5/superficie)',
    fecha_actualizacion DATE COMMENT 'Fecha de actualización de la tabla Gold'
) USING DELTA
PARTITIONED BY (fecha);

In [0]:
# -------------------------------
# 2. Función de escritura flexible
# -------------------------------
def write_gold_fact_contaminacion(df, mode="incremental", table_name="gold.fact_contaminacion_enriquecido"):
    """
    Escribe el DataFrame en la tabla gold de contaminación del aire.
    
    Parámetros:
    - df: DataFrame transformado con columnas ya listas (incluyendo 'fecha').
    - mode: "total" para sobrescribir toda la tabla, "incremental" para sobrescribir solo las particiones afectadas.
    - table_name: nombre de la tabla destino en gold.
    """
    
    writer = df.write.format("delta")#.option("overwriteSchema", "true")
    
    if mode == "total":
        # Sobrescribir toda la tabla
        writer.mode("overwrite").partitionBy("fecha").saveAsTable(table_name)
        print(f"Tabla {table_name} sobrescrita completamente.")
    
    elif mode == "incremental":
        
        # Sobrescribir solo las particiones afectadas
        print(f"Escritura en fechas desde {start_date_str} hasta {end_date_str} en modo {mode}")
        
        writer.mode("overwrite") \
        .option(
            "replaceWhere",
            f"fecha >= DATE'{start_date_str}' AND fecha <= DATE'{end_date_str}'"
        ) \
        .saveAsTable(table_name)

        print(f"Tabla {table_name} sobrescrita solo en fechas desde {start_date_str} hasta {end_date_str} en modo {mode}")
    
    else:
        raise ValueError("El parámetro 'mode' debe ser 'total' o 'incremental'.")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Paso 1: Mapear AQI numérico a string 
# Leer Silver
if tipo_carga == "incremental":
    df_aqi = (
    spark.table("silver.hd_lima_air_pollution").filter((F.col("fecha") >= start_date_str) & (F.col("fecha") <= end_date_str))
    .withColumn(
        "index_air_quality_str",
        F.when(F.col("index_air_quality") == 1, "Bueno")
         .when(F.col("index_air_quality") == 2, "Aceptable")
         .when(F.col("index_air_quality") == 3, "Moderado")
         .when(F.col("index_air_quality") == 4, "Malo")
         .when(F.col("index_air_quality") == 5, "Muy Malo")
         .otherwise("Desconocido")
        )
    )

elif tipo_carga == "total":
    df_aqi = (
    spark.table("silver.hd_lima_air_pollution")
    .withColumn(
        "index_air_quality_str",
        F.when(F.col("index_air_quality") == 1, "Bueno")
         .when(F.col("index_air_quality") == 2, "Aceptable")
         .when(F.col("index_air_quality") == 3, "Moderado")
         .when(F.col("index_air_quality") == 4, "Malo")
         .when(F.col("index_air_quality") == 5, "Muy Malo")
         .otherwise("Desconocido")
        )
    )
    
# Paso 2: Calcular promedios de contaminantes
df_promedios = (
    df_aqi
    .groupBy("fecha", "id_ubigeo")
    .agg(
        F.round(F.avg("conc_pm2_5"),2).alias("cant_prom_pm2_5"),
        F.round(F.avg("conc_pm10"),2).alias("cant_prom_pm10"),
        F.round(F.avg("conc_co"),2).alias("cant_prom_co"),
        F.round(F.avg("conc_no2"),2).alias("cant_prom_no2"),
        F.round(F.avg("conc_so2"),2).alias("cant_prom_so2"),
        F.round(F.avg("conc_o3"),2).alias("cant_prom_o3")
    ).alias("df_promedios")
)

# Paso 3: Calcular la moda del índice de calidad del aire
# Contamos ocurrencias por fecha y distrito
df_moda = (
    df_aqi
    .groupBy("fecha", "id_ubigeo", "index_air_quality_str")
    .agg(F.count("*").alias("freq"))
)

# Definimos ventana para ordenar por frecuencia
w = Window.partitionBy("fecha", "id_ubigeo").orderBy(F.desc("freq"))

df_moda = (
    df_moda
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)  # nos quedamos con el más frecuente
    .select("fecha", "id_ubigeo", F.col("index_air_quality_str").alias("moda_index_air_quality")).alias("df_moda")
)

# Paso 4: Unir promedios con moda
df_fact_contaminacion = (
    df_promedios
    .join(df_moda, ["fecha", "id_ubigeo"], "left")
).select(
    *df_promedios.columns,
    "moda_index_air_quality",
    F.lit(F.current_date()).alias("fecha_actualizacion")
).alias("fact_contaminacion")

In [0]:
df_ubi_equi = spark.table("workspace.silver.m_ubigeo_equivalencia").alias("A")
df_ubi_geo = spark.table("workspace.silver.m_ubigeo_geografia").alias("B")
df_ubi_geo_homologado =  df_ubi_equi.join(df_ubi_geo, "id_ubigeo_reniec", "left").select("A.id_ubigeo", "B.superficie").alias("C")

df_ubi_demo = spark.table("workspace.silver.m_ubigeo_demografia").alias("D")

df_fact_enriquecido = (
    df_fact_contaminacion
    .join(df_ubi_demo, "id_ubigeo", "left")
    .join(df_ubi_geo_homologado, "id_ubigeo", "left")
    .select(
        "fact_contaminacion.fecha",
        "fact_contaminacion.id_ubigeo",
        "fact_contaminacion.cant_prom_pm2_5",
        "fact_contaminacion.cant_prom_pm10",
        "fact_contaminacion.cant_prom_co",
        "fact_contaminacion.cant_prom_no2",
        "fact_contaminacion.cant_prom_so2",
        "fact_contaminacion.cant_prom_o3",
        "fact_contaminacion.moda_index_air_quality",
        "D.poblacion",
        "C.superficie",
        F.round((F.col("D.poblacion") / F.col("C.superficie")),2).alias("densidad_poblacional"),
        F.round(((F.col("fact_contaminacion.cant_prom_pm2_5") * 1000) / F.col("D.poblacion")),2).alias("contaminacion_per_capita_1000"),
        F.round((F.col("fact_contaminacion.cant_prom_pm2_5") / F.col("C.superficie")),2).alias("contaminacion_por_km2"),
        F.lit(F.current_date()).alias("fecha_actualizacion")
    )
)

In [0]:
# -------------------------------
# Escritura en gold
# -------------------------------
# incremental (solo particiones afectadas)
if tipo_carga == "incremental":
    write_gold_fact_contaminacion(df_fact_contaminacion, mode="incremental", table_name="workspace.gold.fact_contaminacion")
    write_gold_fact_contaminacion(df_fact_enriquecido, mode="incremental", table_name="workspace.gold.fact_contaminacion_enriquecido")
    
elif tipo_carga == "total":
    # total (reescribe toda la tabla)
    write_gold_fact_contaminacion(df_fact_contaminacion, mode="total", table_name="workspace.gold.fact_contaminacion")
    write_gold_fact_contaminacion(df_fact_enriquecido, mode="total", table_name="workspace.gold.fact_contaminacion_enriquecido")